# Couche Silver — qualité et préparation des comptages

Ce notebook lit la couche Bronze, contrôle et nettoie les données, puis écrit une table Silver réutilisable pour l'exploration et les agrégations Gold. Les valeurs descriptives manquantes ne provoquent pas la suppression d'un comptage : elles sont complétées lorsque c'est possible, sinon laissées nulles.

## 1. Connexion à Spark et au data lake

Démarrer les services avec `docker compose up -d`, puis exécuter les cellules dans l'ordre. La session et les chemins reprennent les conventions de `commun/spark_utils.py` et de `work/notebook_b.ipynb`.

In [ ]:
import re
import unicodedata

from pyspark.sql import Window
from pyspark.sql import functions as F
from commun.spark_utils import LAKE, creer_session, format_tables

spark = creer_session("S02_Silver", lac=True, delta=True, postgres=False)
spark.conf.set("spark.sql.session.timeZone", "Europe/Paris")
FMT = format_tables(spark)
BRONZE_PATH = f"{LAKE}/bronze/comptage_multimodal_paris"
SILVER_PATH = f"{LAKE}/silver/comptage_multimodal_paris"
print(f"Format des tables : {FMT}")
print(f"Bronze : {BRONZE_PATH}")
print(f"Silver : {SILVER_PATH}")

## 2. Lecture et inspection du Bronze

Les en-têtes sont normalisés pour accepter les accents et les variantes d'encodage observées dans les sorties du notebook Bronze. Une colonne requise absente déclenche une erreur explicite plutôt qu'un nettoyage silencieusement incomplet.

In [ ]:
brut = spark.read.format(FMT).load(BRONZE_PATH)

def normaliser_entete(nom):
    texte = unicodedata.normalize("NFKD", nom).encode("ascii", "ignore").decode("ascii").lower()
    return re.sub(r"_+", "_", re.sub(r"[^a-z0-9]+", "_", texte)).strip("_")

alias = {
    "identifiant_trajectoire": {"identifiant_trajectoire"},
    "identifiant_site": {"identifiant_site"},
    "nom_du_site": {"nom_du_site"},
    "date_heure_source": {"date_et_heure_de_comptage", "date_heure_de_comptage"},
    "mode_source": {"mode_deplacement", "mode_dplacement"},
    "nombre_source": {"nombre_de_vehicules", "nombre_de_vicules", "nombre_de_vhicules"},
    "type_voie": {"type_voie"},
    "sens_de_circulation": {"sens_de_circulation"},
    "trajectoire": {"trajectoire"},
    "coordonnees_geo": {"coordonnees_geo", "coordonnes_geo", "coordonnes_go"},
}
colonnes_normalisees = {normaliser_entete(c): c for c in brut.columns}
renommages = {}
for cible, variantes in alias.items():
    correspondances = [colonnes_normalisees[v] for v in variantes if v in colonnes_normalisees]
    if not correspondances:
        raise ValueError(f"Colonne Bronze requise introuvable pour {cible!r}. Colonnes reçues : {brut.columns}")
    renommages[correspondances[0]] = cible

df = brut
for source, cible in renommages.items():
    if source != cible:
        df = df.withColumnRenamed(source, cible)

brut_canonique = df
nb_depart = df.count()
print(f"Lignes au départ : {nb_depart:,}")
df.printSchema()
df.select(
    F.count("*").alias("lignes"),
    *[F.sum(F.col(c).isNull().cast("long")).alias(c) for c in alias]
).show(truncate=False)
df.select("identifiant_site", "nom_du_site", "date_heure_source", "mode_source", "nombre_source").show(5, truncate=False)

## 3. Typage des champs et préparation du journal

Les conversions tolèrent les valeurs illisibles : elles deviennent nulles et seront comptabilisées par les règles de qualité. Les colonnes textuelles sont débarrassées des espaces superflus avant leur validation.

In [ ]:
df = (df
    .withColumn("date_heure", F.expr("try_to_timestamp(cast(date_heure_source as string))"))
    .withColumn("nombre_vehicules", F.expr("try_cast(nombre_source as BIGINT)")))

for colonne in ["identifiant_trajectoire", "identifiant_site", "nom_du_site", "mode_source",
                "type_voie", "sens_de_circulation", "trajectoire", "coordonnees_geo"]:
    df = df.withColumn(
        colonne,
        F.when(F.length(F.trim(F.col(colonne).cast("string"))) == 0, F.lit(None))
         .otherwise(F.trim(F.col(colonne).cast("string")))
    )

journal = []
lignes_courantes = nb_depart

def appliquer_regle(data, nom, justification, transformation):
    global lignes_courantes
    resultat = transformation(data)
    restantes = resultat.count()
    journal.append({
        "regle": nom,
        "justification": justification,
        "lignes_retirees": lignes_courantes - restantes,
        "lignes_restantes": restantes,
    })
    print(f"{nom} : {lignes_courantes - restantes:,} retirées ; {restantes:,} restantes")
    lignes_courantes = restantes
    return resultat

## 4. Règles de qualité

Seules les lignes impossibles à exploiter pour les analyses (doublon exact, identité/localisation temporelle indispensable absente, mode ou nombre invalide, nombre négatif) sont retirées. Un nom de site, une voie, un sens ou des coordonnées manquants ne suffisent pas à supprimer une observation.

In [ ]:
df = appliquer_regle(
    df, "R1 — doublons exacts", "Une même ligne source ne doit être conservée qu'une fois.",
    lambda d: d.dropDuplicates())
df = appliquer_regle(
    df, "R2 — trajectoire absente", "La trajectoire est nécessaire pour rattacher le comptage à un axe.",
    lambda d: d.filter(F.col("identifiant_trajectoire").isNotNull()))
df = appliquer_regle(
    df, "R3 — date absente ou illisible", "Une date valide est indispensable à l'analyse temporelle.",
    lambda d: d.filter(F.col("date_heure").isNotNull()))
df = appliquer_regle(
    df, "R4 — identifiant de site absent", "Sans identifiant de site, le comptage ne peut pas être localisé.",
    lambda d: d.filter(F.col("identifiant_site").isNotNull()))
df = appliquer_regle(
    df, "R5 — mode absent", "Le mode de déplacement est nécessaire aux comparaisons multimodales.",
    lambda d: d.filter(F.col("mode_source").isNotNull()))
df = appliquer_regle(
    df, "R6 — comptage absent ou non numérique", "Le nombre de véhicules doit pouvoir être interprété comme un entier.",
    lambda d: d.filter(F.col("nombre_vehicules").isNotNull()))
df = appliquer_regle(
    df, "R7 — comptage négatif", "Un nombre de véhicules négatif n'est pas physiquement interprétable.",
    lambda d: d.filter(F.col("nombre_vehicules") >= 0))

print(f"Lignes retenues : {lignes_courantes:,} sur {nb_depart:,}")

## 5. Complétion prudente des attributs descriptifs

Les valeurs manquantes sont récupérées depuis les observations du même site ou de la même trajectoire. En cas de valeurs contradictoires, le mode le plus fréquent est choisi avec un départage déterministe. Les valeurs non retrouvées restent nulles : aucune coordonnée ni description n'est inventée.

In [ ]:
def valeur_la_plus_frequente(data, cle, valeur, alias):
    fenetre = Window.partitionBy(cle).orderBy(F.desc("frequence"), F.asc(valeur))
    return (data.filter(F.col(valeur).isNotNull())
        .groupBy(cle, valeur).count().withColumnRenamed("count", "frequence")
        .withColumn("rang", F.row_number().over(fenetre))
        .filter(F.col("rang") == 1)
        .select(cle, F.col(valeur).alias(alias)))

noms_site = valeur_la_plus_frequente(df, "identifiant_site", "nom_du_site", "nom_site_ref")
coord_site = valeur_la_plus_frequente(df, "identifiant_site", "coordonnees_geo", "coordonnees_ref")
voies_trajet = valeur_la_plus_frequente(df, "identifiant_trajectoire", "type_voie", "type_voie_ref")
sens_trajet = valeur_la_plus_frequente(df, "identifiant_trajectoire", "sens_de_circulation", "sens_ref")

colonnes_a_completer = ["nom_du_site", "coordonnees_geo", "type_voie", "sens_de_circulation"]
manquantes_avant = df.select(*[
    F.sum(F.col(c).isNull().cast("long")).alias(c) for c in colonnes_a_completer
]).first().asDict()

df = (df
    .join(noms_site, "identifiant_site", "left")
    .join(coord_site, "identifiant_site", "left")
    .join(voies_trajet, "identifiant_trajectoire", "left")
    .join(sens_trajet, "identifiant_trajectoire", "left")
    .withColumn("nom_du_site", F.coalesce("nom_du_site", "nom_site_ref"))
    .withColumn("coordonnees_geo", F.coalesce("coordonnees_geo", "coordonnees_ref"))
    .withColumn("type_voie", F.coalesce("type_voie", "type_voie_ref"))
    .withColumn("sens_de_circulation", F.coalesce("sens_de_circulation", "sens_ref"))
    .drop("nom_site_ref", "coordonnees_ref", "type_voie_ref", "sens_ref"))

manquantes_apres = df.select(*[
    F.sum(F.col(c).isNull().cast("long")).alias(c) for c in colonnes_a_completer
]).first().asDict()
valeurs_recuperees = {c: manquantes_avant[c] - manquantes_apres[c] for c in colonnes_a_completer}
journal.append({
    "regle": "R8 — complétion des attributs descriptifs",
    "justification": "Complétion par site/trajectoire ; les inconnues restent nulles.",
    "lignes_retirees": 0,
    "lignes_restantes": lignes_courantes,
})
print("Valeurs descriptives récupérées :", valeurs_recuperees)
df.select(*[
    F.sum(F.col(c).isNull().cast("long")).alias(c) for c in colonnes_a_completer
]).show(truncate=False)

## 6. Normalisation et variables d'analyse

Les libellés de mode sont nettoyés sans fusionner des catégories différentes. Les coordonnées sont séparées en latitude et longitude numériques. `geo_connue` permet d'exclure les lignes non cartographiables sans perdre leurs comptages.

In [ ]:
df = (df
    .withColumn("mode_deplacement", F.regexp_replace(F.trim("mode_source"), r"\s+", " "))
    .withColumn("latitude", F.expr("try_cast(trim(split(coordonnees_geo, ',')[0]) AS DOUBLE)"))
    .withColumn("longitude", F.expr("try_cast(trim(split(coordonnees_geo, ',')[1]) AS DOUBLE)"))
    .withColumn("annee", F.year("date_heure"))
    .withColumn("mois", F.month("date_heure"))
    .withColumn("jour", F.to_date("date_heure"))
    .withColumn("jour_semaine", F.dayofweek("date_heure"))
    .withColumn("heure", F.hour("date_heure"))
    .withColumn("week_end", F.dayofweek("date_heure").isin(1, 7))
    .withColumn("geo_connue", F.col("latitude").between(-90, 90) & F.col("longitude").between(-180, 180)))

df.select("mode_deplacement", "nombre_vehicules", "date_heure", "latitude", "longitude").show(5, truncate=False)
df.groupBy("mode_deplacement").count().orderBy(F.desc("count")).show(truncate=False)

## 7. Repérage exploratoire des valeurs élevées

Le seuil est calculé par mode au 99,9e percentile. C'est un repère exploratoire, pas une preuve d'erreur : les lignes signalées sont conservées et ne sont pas supprimées automatiquement. Les distributions et le contexte métier devront être examinés avant toute exclusion.

In [ ]:
seuils_modes = (df.groupBy("mode_deplacement")
    .agg(F.expr("percentile_approx(nombre_vehicules, 0.999, 10000)").alias("seuil_q999")))
df = (df.join(seuils_modes, "mode_deplacement", "left")
    .withColumn("aberrant_q999", F.col("nombre_vehicules") > F.col("seuil_q999")))
seuils_modes.orderBy("mode_deplacement").show(truncate=False)
df.groupBy("mode_deplacement").agg(
    F.sum(F.col("aberrant_q999").cast("long")).alias("lignes_signalees"),
    F.max("nombre_vehicules").alias("maximum")
).orderBy(F.desc("lignes_signalees")).show(truncate=False)

## 8. Écriture de la table Silver

La sortie utilise le format Delta si la session l'a activé (sinon le format de repli défini par les outils partagés), partitionnée par année. Le comptage relu depuis le data lake est vérifié avant de conclure.

In [ ]:
silver = df.select(
    "identifiant_site", "identifiant_trajectoire", "nom_du_site", "latitude", "longitude",
    "geo_connue", "type_voie", "sens_de_circulation", "trajectoire", "mode_deplacement",
    "nombre_vehicules", "date_heure", "jour", "annee", "mois", "jour_semaine", "heure",
    "week_end", "seuil_q999", "aberrant_q999"
)

(silver.write.format(FMT).mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("annee")
    .save(SILVER_PATH))

silver_verifie = spark.read.format(FMT).load(SILVER_PATH)
nb_silver = silver_verifie.count()
if nb_silver != lignes_courantes:
    raise RuntimeError(f"Comptage Silver incohérent : {nb_silver} relues, {lignes_courantes} attendues")
print(f"Silver écrite et vérifiée : {nb_silver:,} lignes dans {SILVER_PATH}")
silver_verifie.printSchema()

## 9. Bilan des règles

Ces valeurs sont calculées pendant l'exécution et constituent le bilan à reporter dans `docs/silver.md`. Le nombre final doit être égal au nombre de lignes relues dans la table Silver.

In [ ]:
bilan = spark.createDataFrame(
    [(r["regle"], r["justification"], r["lignes_retirees"], r["lignes_restantes"]) for r in journal],
    ["regle", "justification", "lignes_retirees", "lignes_restantes"]
)
bilan.show(truncate=False)
print(f"TOTAL retiré : {nb_depart - nb_silver:,}")
print(f"TOTAL conservé : {nb_silver:,}")

## 10. Avant / après nettoyage

La comparaison ci-dessous montre la somme mensuelle des comptages par mode dans le Bronze brut et dans Silver. Les deux panneaux partagent la même échelle et commencent à zéro pour éviter une comparaison visuellement trompeuse. Le Bronze n'est pas dédupliqué et contient encore les lignes invalides ; Silver applique les règles ci-dessus.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

brut_graph = (brut_canonique
    .withColumn("date_graph", F.expr("try_to_timestamp(cast(date_heure_source as string))"))
    .withColumn("nombre_graph", F.expr("try_cast(nombre_source as BIGINT)"))
    .withColumn("mode_graph", F.regexp_replace(F.trim(F.col("mode_source")), r"\s+", " "))
    .filter(F.col("date_graph").isNotNull() & F.col("mode_graph").isNotNull())
    .withColumn("mois_graph", F.date_trunc("month", "date_graph"))
    .groupBy("mois_graph", "mode_graph")
    .agg(F.sum("nombre_graph").alias("comptages")))

modes_top = [r["mode_deplacement"] for r in
    silver.groupBy("mode_deplacement").agg(F.sum("nombre_vehicules").alias("total"))
    .orderBy(F.desc("total")).limit(5).collect()]
avant = brut_graph.filter(F.col("mode_graph").isin(modes_top)).toPandas()
apres = (silver.withColumn("mois_graph", F.date_trunc("month", "date_heure"))
    .filter(F.col("mode_deplacement").isin(modes_top))
    .groupBy("mois_graph", "mode_deplacement")
    .agg(F.sum("nombre_vehicules").alias("comptages"))
    .toPandas())

if not modes_top or avant.empty or apres.empty:
    raise ValueError("Impossible de tracer la comparaison : aucun mode ou agrégat disponible.")
for table in (avant, apres):
    table["mois_graph"] = pd.to_datetime(table["mois_graph"])
limite_y = max(avant["comptages"].max(), apres["comptages"].max()) * 1.05
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for mode in modes_top:
    a = avant[avant["mode_graph"] == mode].sort_values("mois_graph")
    b = apres[apres["mode_deplacement"] == mode].sort_values("mois_graph")
    axes[0].plot(a["mois_graph"], a["comptages"], label=mode)
    axes[1].plot(b["mois_graph"], b["comptages"], label=mode)
axes[0].set_title("Avant — Bronze brut")
axes[1].set_title("Après — Silver nettoyée")
for ax in axes:
    ax.set_ylim(bottom=0, top=limite_y)
    ax.set_xlabel("Mois")
    ax.grid(axis="y", alpha=0.25)
axes[0].set_ylabel("Somme des véhicules comptés")
axes[1].legend(title="Mode", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.suptitle("Comptages mensuels : données brutes et données Silver (même échelle)")
fig.tight_layout()
plt.show()